# 03. Adaptive scheduling

Reproduces the adaptive scheduling pipeline of Section 6.4 of the report.

The pipeline conditions schedules on hour-level summary statistics of the visible window rather than on per-second prediction. Four families of schedule are considered:

- `fixed_k`               one K for all hours of a pair, alpha = 0
- `fixed_k_alpha`         one (K, alpha) for all hours of a pair
- `adaptive_k`            quantile-bin a single feature, choose K per bucket
- `adaptive_k_alpha`      quantile-bin a single feature, choose (K, alpha) per bucket

An optional capping overlay clips per-second positions and redistributes the clipped volume. Selection is by nested cross-validation on the dev partition; the holdout is touched only at the end as a veto.

In [ ]:
from execution_edge.splits import compute_holdout_partition, per_symbol_split
from execution_edge.preprocessing import normalize_last_minute_frame
from execution_edge.features import build_hour_features_from_x
from execution_edge.candidates import candidate_feature_sets, DEFAULT_CANDIDATES
from execution_edge.schedules import build_twap_schedule, cap_schedule_preserve_support
from execution_edge.selection import sorted_id_folds, cost_matrix_from_hourly_scores
from execution_edge.evaluation import build_hour_books, score_schedule
from execution_edge.walk_the_book import simulate_walk_the_book
from execution_edge.bps import compute_bps
from execution_edge.data import (
    DATA_DIR, load_parquet_split, load_volume_to_fill,
    ASK_PRICE_COLS, ASK_VOL_COLS, BID_PRICE_COLS, BID_VOL_COLS,
)
import numpy as np
import pandas as pd

SYMBOLS = ["BTCUSDT","ETHUSDT","LTCUSDT","SOLUSDT","ADAUSDT","DOGEUSDT","XRPUSDT"]
dev_ids, holdout_ids = compute_holdout_partition(DATA_DIR, SYMBOLS, fraction=0.20)
print(f"dev={len(dev_ids)}, holdout={len(holdout_ids)}")

## 1. Hour-level features

For each hour the pipeline computes summaries over windows of 60, 300, 900, 1800, and 3540 seconds, on top of: relative spread, level-1 ask/bid volumes, level-5 aggregate volumes, top-of-book imbalance, absolute one-second log mid-return, OHLCV trade volume, close- and volume-missingness rates, and microprice. The implementation is in `execution_edge/features.py`.

In [ ]:
# Example: compute features on LTC visible data.
symbol = "LTCUSDT"
vol = load_volume_to_fill(symbol)
x = load_parquet_split(symbol, "X_train")
features = build_hour_features_from_x(x, volume_to_fill=vol, symbol=symbol)
print(f"features: {len(features)} hours, {features.shape[1]} columns")
print(f"sample columns: {[c for c in features.columns if c.startswith('w300_')][:6]}")

## 2. Pipeline structure

The full deployed pipeline is implemented in `execution_edge/selection.py` and runs in four stages:

1. **Feature extraction** (`build_hour_features_from_x` in `features.py`): produces several hundred per-hour features per pair.
2. **Candidate generation** (`candidate_feature_sets` in `candidates.py`): builds the pool of feature combinations to evaluate, gated by quantile bin counts in `{3, 5, 7}` and alpha grids in `{0, 0.5, 1, 2, 4}`.
3. **Stage 1 (model class).** Choose the model class (fixed_k, fixed_(k,alpha), adaptive_k, adaptive_(k,alpha)) for each pair through outer 5-fold cross-validation on the dev partition.
4. **Stage 2 (parameter selection).** For the winning class, run an inner 4-fold cross-validation to pick the feature, bin count, K-per-bucket, and (alpha-per-bucket where applicable). Capping overlay parameters are jointly searched.
5. **Holdout veto.** Compare the Stage-2 winner against a holdout-restricted version of the same search; on disagreement, override to a simpler fixed-shape rule. Two pairs (ADA bid, LTC bid) were overridden under this procedure.

Running the full tournament across all candidate features, bin counts, alpha grids, and capping overlay parameters is several hours of CPU work and is not executed inline; it is what produces the canonical deployed rules in Tables 7 and 8 below.

## 2bis. Runnable demo: one pair, one feature

The cell below runs the simplest possible adaptive scheduling rule end to end: pick one pair-side (LTC ask), pick one feature (`w300_abs_return_sum`) and one bin count (5), build the quantile-bucket model on dev, apply it to the holdout, and score with the walk-the-book simulator.

This is one cell from the full tournament; the canonical Table 7 row for LTC comes from sweeping all candidate features, bin counts {3, 5, 7}, alpha grids {0, 0.5, 1, 2, 4}, and capping overlay parameters, then running the inner four-fold cross-validation, then applying the holdout veto. The cell below skips the sweep and the CV; it just demonstrates the data path from feature extraction to BPS scoring on a single configuration. Runtime is a few minutes.

In [ ]:
import re
from execution_edge.selection import QuantileBucketModel, CandidateSpec, sorted_id_folds

# --- pick one pair-side and one configuration ---
PAIR, SIDE, FEATURE, BINS = "LTCUSDT", "ask", "w300_abs_return_sum", 5

vol = load_volume_to_fill(PAIR)
y = load_parquet_split(PAIR, "y_train")
x = load_parquet_split(PAIR, "X_train")
feats = build_hour_features_from_x(x, volume_to_fill=vol, symbol=PAIR)

# --- 1. dev/holdout split for this pair ---
sym_ids = sorted({int(i) for i in feats["anonymized_id"].astype("uint64").unique()})
dev_sym, held_sym = per_symbol_split(sym_ids, holdout_ids)

dev_feat = feats[feats["anonymized_id"].astype("uint64").isin(set(dev_sym))].copy()
held_feat = feats[feats["anonymized_id"].astype("uint64").isin(set(held_sym))].copy()

# --- 2. Build a cost matrix on dev: per-hour BPS at each K in 1..60 ---
y_dev_norm = normalize_last_minute_frame(y[y["anonymized_id"].isin(dev_sym)])
y_dev_by_id = {int(a): hf for a, hf in y_dev_norm.groupby("anonymized_id", sort=True)
               if not hf["close"].dropna().empty}
print(f"dev hours with usable close: {len(y_dev_by_id)}")

K_VALUES = tuple(range(1, 61))
cost_rows = []
for aid, hf in y_dev_by_id.items():
    close = float(hf["close"].dropna().iloc[-1])
    ask_p = hf[list(ASK_PRICE_COLS)].to_numpy(float)
    ask_v = hf[list(ASK_VOL_COLS)].to_numpy(float)
    bid_p = hf[list(BID_PRICE_COLS)].to_numpy(float)
    bid_v = hf[list(BID_VOL_COLS)].to_numpy(float)
    row = {"anonymized_id": aid}
    for K in K_VALUES:
        sched = build_twap_schedule(total_volume=vol, k=K, side=SIDE, alpha=0.0)
        tot, vwap = simulate_walk_the_book(sched, ask_p, ask_v, bid_p, bid_v)
        b = compute_bps(vwap, close, vol, tot)
        row[K] = float(b) if not np.isnan(b) else np.nan
    cost_rows.append(row)
cost_matrix = pd.DataFrame(cost_rows).sort_values("anonymized_id").reset_index(drop=True)
merged = dev_feat.merge(cost_matrix, on="anonymized_id", how="inner")
print(f"merged feature+cost frame: {merged.shape}")

# --- 3. Fit the quantile-bucket model on dev ---
spec = CandidateSpec(feature_names=(FEATURE,), bins=BINS)
model = QuantileBucketModel(spec=spec, k_values=K_VALUES).fit(merged)
print(f"per-bucket K assignments: {model.bucket_to_k}")

# --- 4. Apply on holdout ---
y_held_norm = normalize_last_minute_frame(y[y["anonymized_id"].isin(held_sym)])
y_held_by_id = {int(a): hf for a, hf in y_held_norm.groupby("anonymized_id", sort=True)
                if not hf["close"].dropna().empty}

bps_list = []
for _, row in held_feat.iterrows():
    aid = int(row["anonymized_id"])
    if aid not in y_held_by_id: continue
    K = int(model.predict_k(pd.DataFrame([row]))[0])
    sched = build_twap_schedule(total_volume=vol, k=K, side=SIDE, alpha=0.0)
    hf = y_held_by_id[aid]
    close = float(hf["close"].dropna().iloc[-1])
    tot, vwap = simulate_walk_the_book(
        sched,
        hf[list(ASK_PRICE_COLS)].to_numpy(float),
        hf[list(ASK_VOL_COLS)].to_numpy(float),
        hf[list(BID_PRICE_COLS)].to_numpy(float),
        hf[list(BID_VOL_COLS)].to_numpy(float),
    )
    b = compute_bps(vwap, close, vol, tot)
    if not np.isnan(b): bps_list.append(b)

mean_bps = float(np.mean(bps_list))
print(f"\n{PAIR} {SIDE} demo holdout BPS: {mean_bps:.4f}")
print(f"Table 7 canonical (full tournament): 4.977")

## 3. Deployed rules and holdout BPS

### Table 7: ask-side deployed rules

| Pair | Rule | Capping | TWAP-K | Adaptive | Delta |
|------|------|---------|--------|----------|-------|
| ADA  | Fixed-K, k=17 | none | 5.586 | **5.586** | 0.0% |
| BTC  | Adaptive-(k,alpha), w300_rel_spread_mean, b=5 | flat, ask_vol_1, q=mean, blend, p=5, lambda=0.75 | 1.274 | **1.066** | -16.3% |
| DOGE | Adaptive-(k,alpha), w900_close_missing_rate, b=5 | none | 4.895 | **4.367** | -10.8% |
| ETH  | Adaptive-(k,alpha), w60_trade_volume_mean, b=3 | flat, top_ask_vol_5, q=80, pure_power, p=10 | 3.032 | **2.527** | -16.7% |
| LTC  | Adaptive-k, w300_abs_return_sum, b=5 | none | 5.324 | **4.977** | -6.5% |
| SOL  | Adaptive-k, w900_rel_spread_std, b=5 | none | 5.607 | **5.132** | -8.5% |
| XRP  | Fixed-(k,alpha), k=51, alpha=4 | volregime, top_ask_vol_5, q=50, pure_power, p=1 | 3.906 | **3.687** | -5.6% |

### Table 8: bid-side deployed rules

| Pair | Rule | Capping | TWAP-K | Adaptive | Delta |
|------|------|---------|--------|----------|-------|
| ADA  | Fixed-K, k=10 (override) | volregime, bid_vol_1, q=mean, current_shape | 6.927 | **6.927** | 0.0% |
| BTC  | Adaptive-(k,alpha), w1800_trade_volume_mean, b=5 | none | 1.292 | **1.186** | -8.2% |
| DOGE | Adaptive-(k,alpha), w300_close_missing_rate, b=5 | none | 5.334 | **4.851** | -9.1% |
| ETH  | Adaptive-(k,alpha), w3540_trade_volume_mean, b=3 | none | 3.068 | **2.799** | -8.8% |
| LTC  | Fixed-(k,alpha), k=21, alpha=0 (override) | none | 7.463 | **7.079** | -5.1% |
| SOL  | Fixed-K, k=11 | none | 5.585 | **5.569** | -0.3% |
| XRP  | Fixed-(k,alpha), k=27, alpha=2 | volregime, top_bid_vol_5, q=50, pure_power, p=10 | 4.198 | **4.018** | -4.3% |

ADA bid and LTC bid are the two override pairs from Section 6.4.4: the nested-CV winner was overridden in favour of a simpler fixed-shape rule after the holdout veto detected a disjoint-sample disagreement.

These numbers are produced by running the full nested cross-validation tournament in `execution_edge/selection.py` with the canonical configuration. The runtime is several CPU-hours; we surface the rules and results here so the deployed structure is visible at a glance.